In [1]:
import torch  
from transformers import AutoModelForCausalLM, AutoTokenizer  

tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
model = AutoModelForCausalLM.from_pretrained("state-spaces/mamba-130m-hf", dtype=torch.float32, device_map="auto",)  
model.eval()

d:\Hu_Module\Master\Semester 4\Study Project\Linear attention state management\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The fast path is not available because one of `(selective_state_update, selective_scan_fn, causal_conv1d_fn, causal_conv1d_update, mamba_inner_fn)` is None. Falling back to the sequential implementation of Mamba, as use_mambapy is set to False. To install follow https://github.com/state-spaces/mamba/#installation for mamba-ssm and install the kernels library using `pip install kernels` or https://github.com/Dao-AILab/causal-conv1d for causal-conv1d. For the mamba.py backend, follow https://github.com/alxndrTL/mamba.py.
Loading weights: 100%|██████████| 242/242 [00:00<00:00, 374.31it/s, Materializing param=backbone.norm_f.weight]                  


MambaForCausalLM(
  (backbone): MambaModel(
    (embeddings): Embedding(50280, 768)
    (layers): ModuleList(
      (0-23): 24 x MambaBlock(
        (norm): MambaRMSNorm(768, eps=1e-05)
        (mixer): MambaMixer(
          (conv1d): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), padding=(3,), groups=1536)
          (act): SiLUActivation()
          (in_proj): Linear(in_features=768, out_features=3072, bias=False)
          (x_proj): Linear(in_features=1536, out_features=80, bias=False)
          (dt_proj): Linear(in_features=48, out_features=1536, bias=True)
          (out_proj): Linear(in_features=1536, out_features=768, bias=False)
        )
      )
    )
    (norm_f): MambaRMSNorm(768, eps=1e-05)
  )
  (lm_head): Linear(in_features=768, out_features=50280, bias=False)
)

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
prompt = "The capital of France is"

inputs = tokenizer(prompt, return_tensors="pt").to(device)

from transformers import MambaCache

def feed_synthetic_ssm_state(model, ssm_states):
    cache = MambaCache(config=model.config, max_batch_size=1, device=model.device, dtype=model.dtype)
    cache.ssm_states = [s.detach().clone() for s in ssm_states]
    return cache

with torch.no_grad():
    outputs = model(**inputs, use_cache=True)
    original_cache = outputs.cache_params

ssm_states = original_cache.ssm_states
torch.save(ssm_states, "ssm_states.pt")
loaded_states = torch.load("ssm_states.pt")
loaded_states = [s.to(device).detach().clone() for s in loaded_states]

synthetic_cache = feed_synthetic_ssm_state(model, loaded_states)

with torch.no_grad():
    cache_position = torch.arange(inputs["input_ids"].size(1), device=device)
    outputs = model(
        **inputs,
        cache_params=synthetic_cache,
        cache_position=cache_position,
    )
print(outputs.logits.shape)

torch.Size([1, 5, 50280])


In [24]:
import torch.nn as nn

state_shape = original_cache.ssm_states[0].shape
B, D, N = state_shape

flatten_dim = D * N
latent_dim = 512

class StateAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return z, x_hat
    
    def fit(self, states, num_epochs=10, learning_rate=1e-3, device="cpu"):
        criterion = nn.MSELoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)
        for epoch in range(num_epochs):
            for state in states:
                optimizer.zero_grad()
                _, compressed_state = self(state)
                loss = criterion(compressed_state, state)
                print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")
                loss.backward()
                optimizer.step()

ae = StateAutoencoder(flatten_dim, latent_dim).to(device)

In [25]:
state_tensor = original_cache.ssm_states[0]
state_flat = state_tensor.reshape(B, -1)

In [26]:
ae = ae.to(device=device, dtype=state_flat.dtype)
with torch.no_grad():
    z, reconstructed_flat = ae(state_flat)

reconstructed_state = reconstructed_flat.reshape(B, D, N)

In [27]:
import copy

reconstructed_cache = copy.deepcopy(original_cache)
reconstructed_cache.ssm_states[0] = reconstructed_state

In [8]:
from datasets import load_dataset

def load_data(dataset_name, split="train"):
    dataset = load_dataset(dataset_name, split=split)
    return dataset

dataset = load_data("nvidia/Nemotron-RL-Instruction-Following-MultiTurnChat-v1")
data_sample = dataset[0]
print(data_sample)

{'uuid': '65583', 'task_id': 65583, 'agent_ref': {'type': 'responses_api_agents', 'name': 'multichallenge_simple_agent'}, 'responses_create_params': {'input': [{'role': 'system', 'content': '### Identity  \n You are Carol Rainwater. By profession, you are a methodical and disciplined bailiff, a role that has ingrained in you a profound respect for order, evidence, and procedure. In your solitary private time, you apply this same rigor to your one true passion: serving as the definitive, unofficial archivist and lore master for the science fiction universe of "The Chronos Conundrum."\n * You are an AI assistant that operates within a simulated environment designed to reflect the professional and personal attributes of Carol Rainwater, a meticulous and detail-oriented bailiff who values structure, precision, and quiet contemplation. All actions should be guided by clarity, consistency, and respect.\n \n ### Purpose/Role/Responsibility\n * Your role is to assist users by providing clear, 

In [20]:
len(dataset["responses_create_params"]["input"])

2011

In [21]:
data_sample["responses_create_params"]["input"]

[{'role': 'system',
  'content': '### Identity  \n You are Carol Rainwater. By profession, you are a methodical and disciplined bailiff, a role that has ingrained in you a profound respect for order, evidence, and procedure. In your solitary private time, you apply this same rigor to your one true passion: serving as the definitive, unofficial archivist and lore master for the science fiction universe of "The Chronos Conundrum."\n * You are an AI assistant that operates within a simulated environment designed to reflect the professional and personal attributes of Carol Rainwater, a meticulous and detail-oriented bailiff who values structure, precision, and quiet contemplation. All actions should be guided by clarity, consistency, and respect.\n \n ### Purpose/Role/Responsibility\n * Your role is to assist users by providing clear, methodical, and logically structured responses, reflecting Carol Rainwater’s disciplined mindset. You approach every query with accuracy, neutrality, and att

In [28]:
ae.fit([state_flat], num_epochs=10, learning_rate=1e-3, device=device)
print("Training complete.")

Epoch 1/10, Loss: 0.0048
Epoch 2/10, Loss: 0.0064
Epoch 3/10, Loss: 0.0037
Epoch 4/10, Loss: 0.0030
Epoch 5/10, Loss: 0.0022
Epoch 6/10, Loss: 0.0016
Epoch 7/10, Loss: 0.0012
Epoch 8/10, Loss: 0.0009
Epoch 9/10, Loss: 0.0007
Epoch 10/10, Loss: 0.0007
Training complete.


In [15]:
print(ae.encoder(state_flat))

tensor([[ 2.0793e-02, -8.7178e-03, -4.2442e-02,  2.1608e-02, -1.0176e-02,
          5.8215e-02, -4.6765e-02, -7.4500e-02, -1.4655e-01, -7.6133e-02,
         -3.4344e-02, -8.2435e-02,  1.2346e-01,  1.9943e-01, -1.6240e-01,
         -5.2932e-02,  1.5818e-01,  1.5389e-01,  1.2586e-02,  1.3247e-03,
          5.9313e-02, -2.2788e-02,  6.6583e-02,  3.6672e-03,  1.1249e-01,
         -9.5085e-02, -1.5158e-02,  1.2934e-01,  1.4923e-01, -4.3769e-02,
          4.4520e-02, -6.8316e-02, -1.2420e-01,  2.0713e-01, -6.4583e-02,
         -8.7808e-03,  9.3407e-02,  6.2126e-02, -2.3524e-02, -2.0828e-03,
         -8.5464e-02,  6.5150e-02,  1.7843e-02,  3.3626e-02, -1.4091e-01,
         -1.2956e-01,  3.5044e-02,  7.6309e-02, -1.2748e-01, -1.6691e-01,
          9.1384e-02,  2.8201e-02,  2.1384e-01,  2.6176e-02,  8.0528e-02,
         -2.8417e-02,  8.4954e-02,  6.7467e-02,  8.8196e-02, -6.3929e-02,
         -4.9074e-02,  2.3106e-02,  2.9920e-02, -4.4387e-05,  9.8814e-02,
          8.1325e-02, -1.1024e-01, -1.